# 05 — Compare held-out TEMPTED and MEFISTO performance

This notebook creates report tables.

Generated reports are saved under `data/reports/<timestamp>/`.

In [1]:
from datetime import datetime
from pathlib import Path
import pandas as pd

PRIMARY_METRIC = "balanced_accuracy"
EXPECTED_BATCHES = 20
EXPECTED_DIMENSIONS = 5
root = Path(".") if Path("data").exists() else Path("..")


def completed_runs(method):
    runs = []
    base = root / "data" / method
    if not base.exists():
        return runs

    for run in sorted(base.iterdir(), reverse=True):
        metrics_file = run / "all_metrics.csv"
        predictions_file = run / "all_predictions.csv.gz"
        if not run.is_dir() or not metrics_file.exists() or not predictions_file.exists():
            continue

        metrics = pd.read_csv(metrics_file, dtype={"split_run": str})
        if "split_run" not in metrics or metrics["split_run"].nunique() != 1:
            continue
        if metrics["batch"].nunique() != EXPECTED_BATCHES:
            continue

        batches = sorted(p for p in run.glob("batch_*") if p.is_dir())
        if len(batches) != EXPECTED_BATCHES:
            continue

        valid = True
        for batch in batches:
            score_file = batch / "test_subject_scores.csv.gz"
            if not score_file.exists():
                valid = False
                break
            columns = pd.read_csv(score_file, nrows=1).columns
            factors = [c for c in columns if c.startswith("factor_")]
            if len(factors) != EXPECTED_DIMENSIONS:
                valid = False
                break

        if valid:
            runs.append((run, str(metrics["split_run"].iloc[0])))
    return runs


tempted_runs = completed_runs("tempted")
mefisto_runs = completed_runs("mefisto")
common_split_runs = {split for _, split in tempted_runs} & {split for _, split in mefisto_runs}
if not common_split_runs:
    raise FileNotFoundError("No completed TEMPTED and MEFISTO runs use the same split set.")

# Prefer the newest pair that shares an explicit split-run identifier.
for tempted, split_run in tempted_runs:
    match = next((run for run, split in mefisto_runs if split == split_run), None)
    if match is not None:
        mefisto = match
        break

output = root / "data" / "reports" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

metrics = pd.concat([
    pd.read_csv(tempted / "all_metrics.csv", dtype={"split_run": str}),
    pd.read_csv(mefisto / "all_metrics.csv", dtype={"split_run": str}),
], ignore_index=True)
predictions = pd.concat([
    pd.read_csv(tempted / "all_predictions.csv.gz", dtype={"split_run": str, "subject_id": str}),
    pd.read_csv(mefisto / "all_predictions.csv.gz", dtype={"split_run": str, "subject_id": str}),
], ignore_index=True)

if metrics["split_run"].nunique() != 1 or metrics["split_run"].iloc[0] != split_run:
    raise ValueError("Model runs do not share one split-run identifier.")

batches = sorted(metrics["batch"].unique())
if len(batches) != EXPECTED_BATCHES:
    raise ValueError(f"Expected {EXPECTED_BATCHES} paired batches; found {len(batches)}.")

for batch in batches:
    t = (predictions[(predictions.batch == batch) & (predictions.method == "TEMPTED")]
         [["subject_id", "truth"]].sort_values("subject_id").reset_index(drop=True))
    m = (predictions[(predictions.batch == batch) & (predictions.method == "MEFISTO")]
         [["subject_id", "truth"]].sort_values("subject_id").reset_index(drop=True))
    if not t.equals(m):
        raise ValueError(f"{batch}: TEMPTED and MEFISTO do not contain identical held-out subjects/truth labels.")

print("Split run:", split_run)
print("TEMPTED:", tempted)
print("MEFISTO:", mefisto)


Split run: 20260809_010356
TEMPTED: ../data/tempted/20260809_010443
MEFISTO: ../data/mefisto/20260809_011140


In [2]:
metric_columns = ["accuracy", "balanced_accuracy", "macro_f1"]
summary = metrics.groupby("method")[metric_columns].agg(["count", "mean", "std", "median"]).round(4)
paired = metrics.pivot(index="batch", columns="method", values=PRIMARY_METRIC)
if paired.isna().any().any() or len(paired) != EXPECTED_BATCHES:
    raise ValueError("Performance table is not a complete 20-batch paired comparison.")
paired["TEMPTED_minus_MEFISTO"] = paired["TEMPTED"] - paired["MEFISTO"]
confusion = predictions.groupby(["method", "truth", "predicted"]).size().rename("count").reset_index()

metrics.to_csv(output / "all_metrics.csv", index=False)
summary.to_csv(output / "metric_summary.csv")
paired.reset_index().to_csv(output / "paired_batches.csv", index=False)
confusion.to_csv(output / "confusion_counts.csv", index=False)
(output / "comparison_report.html").write_text(
    "<h1>TEMPTED and MEFISTO comparison</h1>"
    + f"<p>Split run: {split_run}; paired batches: {EXPECTED_BATCHES}; latent dimensions: 5 per method.</p>"
    + "<h2>Summary</h2>" + summary.to_html()
    + f"<h2>Paired {PRIMARY_METRIC}</h2>" + paired.to_html()
    + "<h2>All batches</h2>" + metrics.to_html(index=False), encoding="utf-8")
print("Saved:", output)
summary


Saved: ../data/reports/20260809_153510


accuracy                         balanced_accuracy                  \
           count    mean     std  median             count    mean     std   
method                                                                       
MEFISTO       20  0.6738  0.0551  0.6746                20  0.6738  0.0551   
TEMPTED       20  0.6722  0.0473  0.6667                20  0.6722  0.0473   

                macro_f1                          
         median    count    mean     std  median  
method                                            
MEFISTO  0.6746       20  0.6713  0.0545  0.6746  
TEMPTED  0.6667       20  0.6706  0.0475  0.6665